# MAPPO

> MAPPO (Yu et al., 2022): PPO with a decentralized, parameter-shared actor and a centralized critic, trained online on a pool of envs (torch only).

In [ ]:
#| default_exp wm.mappo.core

In [ ]:
#| hide
from nbdev.showdoc import *

[MAPPO](https://arxiv.org/abs/2103.01955) ("The Surprising Effectiveness of PPO in Cooperative
Multi-Agent Games", Yu et al., NeurIPS 2022 Datasets and Benchmarks) trains the agents with PPO:
a **decentralized actor** shared by the agents (each acts on its own observation) and a
**centralized critic** used only in training (centralized training, decentralized execution). This
implementation follows the paper's recommendations:

* **the critic's input** (`critic`): `'concat'`, every agent's observation (the paper's "CL" global
  state), `'state'`, the env's global state (`state`, the grid with the agents and the goal), or
  `'local'`, the agent's own observation only (IPPO, the paper's baseline); with
  `agent_specific`, the agent's own observation and its one-hot index are appended (the paper's
  agent-specific global state, "AS");
* **value normalization**: the critic predicts returns normalized by running statistics (the
  paper's `ValueNorm`), with a clipped Huber value loss;
* **death masking**: agents that reached their goal act no more; their steps are left out of the
  losses, and their observations out of the others' critic inputs (zeros);
* GAE (`gamma` 0.99, `lam` 0.95), clipped ratios (0.2), an entropy bonus (0.01), several epochs per
  batch, gradient clipping (10), Adam with eps 1e-5, orthogonal initialization;
* episodes end when **every** agent is done (or at the time limit, where the value of the last
  state is bootstrapped), and each env gets a new seed every update: every episode is a new layout.

The observations are the features of the other baselines (`commnet_features`: each agent's
symbolic view `image`, its `direction`, and its offset to its goal). The agents only communicate
for the critic, in training: with `'concat'`, agent `i`'s critic gets agent `k`'s observation only if
it reaches it (alive agents, `comm_range`, `drop_prob`, or comm-core networks: `links`, see
`NetworkLinks`), zeros and a flag otherwise. Acting needs no communication.

In [ ]:
#| export
from __future__ import annotations

from pathlib import Path
from typing import Any, Callable, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from fastcore.utils import patch

from stable_marl.wm.commnet.core import NetworkLinks, comm_mask, commnet_features
from stable_marl.world.env_pool import episode_done

### Value normalization

MAPPO's `ValueNorm`: running mean and mean square of the returns (exponential moving averages with
`beta` close to 1, debiased), used to normalize the critic's targets.

In [ ]:
#| export
class ValueNorm(nn.Module):
    "Running normalization of returns (MAPPO's `ValueNorm`)."
    def __init__(self, beta: float = 0.99999, epsilon: float = 1e-5):
        super().__init__()
        self.beta, self.epsilon = beta, epsilon
        self.register_buffer('mean', torch.zeros(()))
        self.register_buffer('mean_sq', torch.zeros(()))
        self.register_buffer('debias', torch.zeros(()))

    def stats(self) -> tuple[torch.Tensor, torch.Tensor]:
        mean = self.mean / self.debias.clamp(min=self.epsilon)
        var = (self.mean_sq / self.debias.clamp(min=self.epsilon) - mean ** 2).clamp(min=1e-2)
        return mean, var

    @torch.no_grad()
    def update(self, x: torch.Tensor):
        "Add a batch of returns."
        x = x.detach().float()
        self.mean.mul_(self.beta).add_(x.mean() * (1 - self.beta))
        self.mean_sq.mul_(self.beta).add_((x ** 2).mean() * (1 - self.beta))
        self.debias.mul_(self.beta).add_(1 - self.beta)

    def normalize(self, x: torch.Tensor) -> torch.Tensor:
        mean, var = self.stats()
        return (x - mean) / var.sqrt()

    def denormalize(self, x: torch.Tensor) -> torch.Tensor:
        mean, var = self.stats()
        return x * var.sqrt() + mean

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail
norm = ValueNorm(beta=0.9)
for _ in range(200):
    norm.update(torch.randn(256) * 3 + 5)
mean, var = norm.stats()
test_close(mean, 5, eps=0.3); test_close(var.sqrt(), 3, eps=0.3)
x = torch.randn(10)
test_close(norm.denormalize(norm.normalize(x)), x, eps=1e-5)

### The actor and the critic

In [ ]:
#| export
CRITICS = ('concat', 'state', 'local')


def _mlp(sizes: Sequence[int], out_gain: float) -> nn.Sequential:
    "ReLU MLP with orthogonal initialization (gain sqrt(2), `out_gain` for the last layer), its input layer-normalized."
    layers: list[nn.Module] = [nn.LayerNorm(sizes[0])]
    for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
        linear = nn.Linear(a, b)
        nn.init.orthogonal_(linear.weight, out_gain if i == len(sizes) - 2 else np.sqrt(2))
        nn.init.zeros_(linear.bias)
        layers += [linear] if i == len(sizes) - 2 else [linear, nn.ReLU()]
    return nn.Sequential(*layers)


class MAPPO(nn.Module):
    "MAPPO's actor (decentralized, shared by the agents) and critic (centralized, used in training)."
    def __init__(
        self,
        obs_dim: int, # Features per agent (see `commnet_features`)
        num_agents: int,
        num_actions: int = 4,
        hidden_dim: int = 64, # Width of the actor's and critic's MLPs (MAPPO: 64)
        num_layers: int = 2, # Hidden layers of each
        critic: str = 'concat', # The critic's input: 'concat' (all observations), 'state' (the env's state), 'local' (IPPO)
        agent_specific: bool = True, # Append the agent's own observation and one-hot index to a centralized critic's input
        state_dim: int | None = None, # Numbers in the env's `state` (critic='state')
        obs_keys: Sequence[str] = ('image', 'direction'), # Each agent's observations
        goal: bool = True, # Observe the offset to the goal cell
        grid_size: int = 15,
        comm_range: int | None = None, # 'concat': the critic gets the observations of the agents within this many cells
        drop_prob: float = 0.0, # 'concat': probability that an observation does not reach a critic
        bits_per_element: int = 32, # Size of the shared observations for comm-core (as `CommunicationWrapper`: float32)
    ):
        super().__init__()
        if critic not in CRITICS:
            raise ValueError(f"critic must be one of {CRITICS}, got {critic!r}")
        if critic == 'state' and not state_dim:
            raise ValueError("critic='state' needs state_dim")
        self.config = dict(obs_dim=obs_dim, num_agents=num_agents, num_actions=num_actions, hidden_dim=hidden_dim,
                           num_layers=num_layers, critic=critic, agent_specific=agent_specific, state_dim=state_dim,
                           obs_keys=list(obs_keys), goal=goal, grid_size=grid_size, comm_range=comm_range,
                           drop_prob=drop_prob, bits_per_element=bits_per_element)
        self.num_agents, self.critic_input = num_agents, critic
        self.agent_specific = agent_specific and critic != 'local'
        self.obs_keys, self.goal, self.grid_size = tuple(obs_keys), goal, grid_size
        self.comm_range, self.drop_prob, self.bits_per_element = comm_range, drop_prob, bits_per_element
        critic_dim = {'concat': num_agents * (obs_dim + 1), 'state': state_dim or 0, 'local': obs_dim}[critic]
        critic_dim += obs_dim + num_agents if self.agent_specific else 0
        hidden = [hidden_dim] * num_layers
        self.actor = _mlp([obs_dim, *hidden, num_actions], out_gain=0.01)
        self.critic = _mlp([critic_dim, *hidden, 1], out_gain=1.0)
        self.value_norm = ValueNorm()
        self.temperature = 0.0     # acting: 0 picks the most likely action, otherwise samples
        self.checkpoint: dict = {}  # what `save` stored besides the model (e.g. the env)
        self.links: NetworkLinks | None = None   # 'concat': the observations sent to the critics over comm-core

    def critic_inputs(
        self,
        obs: torch.Tensor, # (B, N, obs_dim)
        received: torch.Tensor | None = None, # (B, N, N) bool: agent i's critic got agent k's observation ('concat')
        state: torch.Tensor | None = None, # (B, state_dim) ('state')
    ) -> torch.Tensor:
        "Each agent's critic input (B, N, critic_dim)."
        B, N, _ = obs.shape
        if self.critic_input == 'local':
            return obs
        if self.critic_input == 'concat':
            received = torch.ones(B, N, N, dtype=torch.bool, device=obs.device) if received is None else received.bool()
            got = received.unsqueeze(-1).to(obs.dtype)                                # (B, N, N, 1)
            x = torch.cat([obs.unsqueeze(1) * got, got], -1).flatten(2)               # [:, i] = k's obs (or 0) and flag
        else:
            x = state.float().unsqueeze(1).expand(B, N, -1)
        if self.agent_specific:
            ids = torch.eye(N, device=obs.device).expand(B, N, N)
            x = torch.cat([x, obs, ids], -1)
        return x

    def forward(self, obs: torch.Tensor, critic_input: torch.Tensor | None = None):
        "Action logits (B, N, num_actions) and, with `critic_input`, normalized values (B, N)."
        logits = self.actor(obs)
        return logits, (None if critic_input is None else self.critic(critic_input).squeeze(-1))

In [ ]:
#| hide
torch.manual_seed(0)
obs = torch.randn(2, 3, 5)
for critic, dim in (('concat', 3 * 6 + 5 + 3), ('local', 5), ('state', 7 + 5 + 3)):
    net = MAPPO(5, 3, critic=critic, state_dim=7)
    x = net.critic_inputs(obs, state=torch.randn(2, 7))
    test_eq(x.shape, (2, 3, dim))
    logits, value = net(obs, x)
    test_eq(logits.shape, (2, 3, 4)); test_eq(value.shape, (2, 3))
test_fail(lambda: MAPPO(5, 3, critic='state'))
net = MAPPO(5, 3)
received = torch.ones(2, 3, 3, dtype=torch.bool); received[:, 0, 1] = False
x = net.critic_inputs(obs, received)
test_eq(x[:, 0, 6:12], torch.zeros(2, 6))              # agent 1's observation and flag did not reach agent 0's critic
test_close(x[:, 1, 6:11], obs[:, 1]); test_eq(x[:, 1, 11], torch.ones(2))
test_close(net(obs)[0], net.actor(obs))                # acting uses the actor only

### Observing and acting

`observe` turns columns of one step (`(B, N, ...)`) into the actor's observations, the critics'
inputs and what reached them, and the agents still acting. As the model of a `FeedForwardPolicy`,
`get_action` runs the actor only (decentralized execution).

In [ ]:
#| export
@patch
def _info_keys(self: MAPPO) -> list[str]:
    keys = [*self.obs_keys, 'terminated', 'position']
    if self.goal:
        keys.append('goal_position')
    if self.critic_input == 'state':
        keys.append('state')
    return list(dict.fromkeys(keys))


@patch
def observe(self: MAPPO, columns: dict[str, np.ndarray], critic: bool = True) -> dict[str, torch.Tensor]:
    "`obs` (B, N, obs_dim), `alive` (B, N) and, with `critic`, `critic_input` (B, N, critic_dim) of columns of one step."
    obs = torch.from_numpy(commnet_features(columns, self.obs_keys, self.grid_size, self.goal))
    alive = ~np.asarray(columns['terminated'], bool)
    out = {'obs': obs * torch.from_numpy(alive).unsqueeze(-1), 'alive': torch.from_numpy(alive)}   # death masking
    if not critic:
        return out
    received, state = None, None
    if self.critic_input == 'concat':
        position = np.asarray(columns['position'])
        received = comm_mask(out['alive'], torch.as_tensor(position) if self.comm_range is not None else None,
                             self.comm_range, self.drop_prob)
        if self.links is not None:
            bits = self.bits_per_element * sum(int(np.prod(np.asarray(columns[k]).shape[2:])) for k in self.obs_keys)
            received = received & self.links.masks(position, alive, 1, 0, message_bits=[bits])[:, 0]
        received = received | torch.eye(len(obs[0]), dtype=torch.bool)                     # its own, always
    elif self.critic_input == 'state':
        s = np.asarray(columns['state'])
        state = torch.from_numpy(s.reshape(len(s), -1).astype(np.float32) / 10.0)
    out['critic_input'] = self.critic_inputs(out['obs'], received, state)
    return out


@patch
@torch.no_grad()
def get_action(self: MAPPO, info: dict) -> torch.Tensor:
    "Actions (num_envs, num_agents) of the actor for the info dict of a pool of envs."
    columns = {k: torch.as_tensor(info[k])[:, -1].cpu().numpy() for k in self._info_keys() if k != 'state'}
    obs = self.observe(columns, critic=False)['obs'].to(next(self.parameters()).device)
    logits, _ = self(obs)
    if self.temperature > 0:
        return torch.distributions.Categorical(logits=logits / self.temperature).sample()
    return logits.argmax(-1)

### Rollouts and advantages

`rollout` plays one episode in each env of a `World` with sampled actions. An agent's episode ends
when it reaches its goal (terminal: no value after it) or with the env's time limit (truncation:
the value of its last state is bootstrapped). With `team_reward` (cooperative MAPPO), every agent's
reward is the sum of the agents' rewards. `gae` computes the advantages and returns.

In [ ]:
#| export
def rollout(model: MAPPO, world, seed: int, team_reward: bool = True) -> tuple[dict[str, torch.Tensor], dict[str, np.ndarray]]:
    """
    One episode per env: steps (`obs`, `critic_input`, `alive`, `action`, `log_prob`, `value`, `reward`,
    `terminal`: (T, E, N, ...)), `ended` (T, E: the env's episode ended at this step) and `last_value`
    (E, N: the value of each env's last state); per env `success`, `return` and `length`.
    """
    E = world.num_envs
    world.reset(seed=[seed + i for i in range(E)])
    if model.links is not None:
        model.links.reset(range(E), [seed + i for i in range(E)])
    running = np.ones(E, bool)
    reached = np.zeros((E, world.num_agents), bool)
    length = np.zeros(E, np.int64)
    total = np.zeros(E)
    keys = ('obs', 'critic_input', 'alive', 'action', 'log_prob', 'value', 'reward', 'terminal', 'ended')
    steps: dict[str, list] = {k: [] for k in keys}
    device = next(model.parameters()).device
    columns_of = lambda: {k: np.asarray(world.infos[k])[:, -1].copy() for k in model._info_keys()}
    while running.any():
        columns = columns_of()
        columns['terminated'] = columns['terminated'] | ~running[:, None]
        inputs = model.observe(columns)
        with torch.no_grad():
            logits, value = model(inputs['obs'].to(device), inputs['critic_input'].to(device))
            dist = torch.distributions.Categorical(logits=logits)
            action = dist.sample()
            log_prob, value = dist.log_prob(action).cpu(), model.value_norm.denormalize(value).cpu()
        action = action.cpu()
        _, rewards, terms, truncs, world.infos = world.envs.step(action.numpy(), mask=running)
        alive = inputs['alive']
        reward = torch.from_numpy(rewards)
        if team_reward:
            reward = reward.sum(-1, keepdim=True).expand_as(reward).clone()
        ended = running & episode_done(terms, truncs)
        for k, v in (('obs', inputs['obs']), ('critic_input', inputs['critic_input']), ('alive', alive),
                     ('action', action), ('log_prob', log_prob), ('value', value), ('reward', reward * alive),
                     ('terminal', torch.from_numpy(terms) & alive), ('ended', torch.from_numpy(ended))):
            steps[k].append(v)
        reached |= terms & running[:, None]
        total += rewards.sum(-1) * running
        length += running
        running &= ~ended
    columns = columns_of()                         # each env's last state (envs are not stepped once ended)
    with torch.no_grad():
        inputs = model.observe(columns)
        _, last = model(inputs['obs'].to(device), inputs['critic_input'].to(device))
    batch = {k: torch.stack(v) for k, v in steps.items()}
    batch['last_value'] = model.value_norm.denormalize(last).cpu()
    stats = {'success': reached.all(1), 'return': total, 'length': length}
    return batch, stats


def gae(batch: dict[str, torch.Tensor], gamma: float = 0.99, lam: float = 0.95) -> tuple[torch.Tensor, torch.Tensor]:
    "Advantages and returns (T, E, N) of `rollout`'s steps; zero where agents do not act."
    value, reward, alive = batch['value'], batch['reward'], batch['alive'].float()
    T = len(value)
    next_alive = torch.cat([alive[1:], torch.zeros_like(alive[:1])])
    # the value after a step: the next step's if the agent acts again; at the end of the env's episode, the
    # last state's for agents cut by the time limit; nothing after a terminal step
    truncated = batch['ended'].unsqueeze(-1) & ~batch['terminal'] & batch['alive']
    next_value = torch.cat([value[1:], torch.zeros_like(value[:1])]) * next_alive + batch['last_value'] * truncated
    advantage = torch.zeros_like(value)
    running = torch.zeros_like(value[0])
    for t in reversed(range(T)):
        delta = reward[t] + gamma * next_value[t] - value[t]
        running = delta + gamma * lam * next_alive[t] * running
        advantage[t] = running * alive[t]
    return advantage, (advantage + value) * alive

In [ ]:
#| hide
# a hand-made episode: one env, 2 agents; agent 0 reaches its goal at step 1, agent 1 is cut by the time limit at step 2
batch = {'value': torch.tensor([[[1., 2.]], [[3., 4.]], [[0., 5.]]]),
         'reward': torch.tensor([[[0., 0.]], [[1., 0.]], [[0., 0.]]]),
         'alive': torch.tensor([[[True, True]], [[True, True]], [[False, True]]]),
         'terminal': torch.tensor([[[False, False]], [[True, False]], [[False, False]]]),
         'ended': torch.tensor([[False], [False], [True]]),
         'last_value': torch.tensor([[0., 10.]])}
adv, ret = gae(batch, gamma=0.5, lam=1.0)
# agent 0: step 1 is terminal: delta1 = 1 - 3 = -2; step 0: delta0 = 0 + .5 * 3 - 1 = .5, adv0 = .5 + .5 * -2 = -.5
test_close(adv[:, 0, 0], torch.tensor([-0.5, -2.0, 0.0]))
# agent 1: bootstrapped from the last state's value: delta2 = .5 * 10 - 5 = 0, delta1 = .5 * 5 - 4 = -1.5, delta0 = .5 * 4 - 2 = 0
test_close(adv[:, 0, 1], torch.tensor([0 + .5 * -1.5, -1.5, 0.0]))
test_close(ret, (adv + batch['value']) * batch['alive'])

### Training

In [ ]:
#| export
def ppo_losses(
    model: MAPPO,
    batch: dict[str, torch.Tensor], # Flat samples (M, ...): obs, critic_input, action, log_prob, value, advantage, return
    clip: float = 0.2, # PPO's ratio clipping (and the value clipping)
    entropy_coef: float = 0.01,
    value_coef: float = 1.0,
    huber_delta: float = 10.0, # Huber loss of the values (MAPPO: 10)
) -> dict[str, torch.Tensor]:
    "`loss` (to minimize), `policy_loss`, `value_loss`, `entropy`, `clip_fraction` of a minibatch."
    logits = model.actor(batch['obs'])
    dist = torch.distributions.Categorical(logits=logits)
    ratio = torch.exp(dist.log_prob(batch['action']) - batch['log_prob'])
    adv = batch['advantage']
    policy_loss = -torch.min(ratio * adv, ratio.clamp(1 - clip, 1 + clip) * adv).mean()
    value = model.critic(batch['critic_input']).squeeze(-1)
    target = model.value_norm.normalize(batch['return'])
    old = model.value_norm.normalize(batch['value'])
    clipped = old + (value - old).clamp(-clip, clip)
    value_loss = torch.max(F.huber_loss(value, target, reduction='none', delta=huber_delta),
                           F.huber_loss(clipped, target, reduction='none', delta=huber_delta)).mean()
    entropy = dist.entropy().mean()
    return {'loss': policy_loss + value_coef * value_loss - entropy_coef * entropy, 'policy_loss': policy_loss.detach(),
            'value_loss': value_loss.detach(), 'entropy': entropy.detach(),
            'clip_fraction': ((ratio - 1).abs() > clip).float().mean().detach()}


def train_mappo(
    model: MAPPO,
    world, # A `World`: every update plays one episode per env
    episodes: int = 20_000, # Episodes played in total (updates: episodes / world.num_envs)
    epochs: int = 10, # PPO epochs per batch (MAPPO: 5-15)
    minibatches: int = 1, # Minibatches per epoch (MAPPO: 1)
    lr: float = 5e-4,
    clip: float = 0.2,
    entropy_coef: float = 0.01,
    value_coef: float = 1.0,
    gamma: float = 0.99,
    lam: float = 0.95,
    max_grad_norm: float = 10.0,
    normalize_advantage: bool = True,
    team_reward: bool = True, # Every agent's reward is the team's (cooperative MAPPO)
    seed: int = 0, # Seeds of the episodes: seed, seed + 1, ...
    log_every: int = 20, # Updates between two `log` calls
    log: Callable[[dict], None] | None = None, # Called with the episodes played and the means since the last call
) -> MAPPO:
    "Train `model` online with MAPPO; returns it in eval mode."
    device = next(model.parameters()).device
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, eps=1e-5)
    E = world.num_envs
    totals: dict[str, float] = {}
    count = 0
    for update in range(1, episodes // E + 1):
        model.eval()
        batch, stats = rollout(model, world, seed + (update - 1) * E, team_reward)
        advantage, returns = gae(batch, gamma, lam)
        valid = batch['alive']
        flat = {k: batch[k][valid] for k in ('obs', 'critic_input', 'action', 'log_prob', 'value')}
        flat['advantage'], flat['return'] = advantage[valid], returns[valid]
        if normalize_advantage and len(flat['advantage']) > 1:
            flat['advantage'] = (flat['advantage'] - flat['advantage'].mean()) / (flat['advantage'].std() + 1e-5)
        flat = {k: v.to(device) for k, v in flat.items()}
        model.value_norm.update(flat['return'])
        model.train()
        row: dict[str, float] = {}
        M = len(flat['action'])
        for _ in range(epochs):
            for idx in torch.randperm(M, device=device).chunk(minibatches):
                losses = ppo_losses(model, {k: v[idx] for k, v in flat.items()}, clip, entropy_coef, value_coef)
                optimizer.zero_grad(set_to_none=True)
                losses['loss'].backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                for k, v in losses.items():
                    row[k] = row.get(k, 0.0) + float(v) / (epochs * minibatches)
        row.update(success_rate=float(stats['success'].mean()), episode_return=float(stats['return'].mean()),
                   length=float(stats['length'].mean()))
        for k, v in row.items():
            totals[k] = totals.get(k, 0.0) + v
        count += 1
        if log is not None and (update % log_every == 0 or update == episodes // E):
            log({'episodes': update * E, **{k: v / count for k, v in totals.items()}})
            totals, count = {}, 0
    return model.eval()

### Saving and loading

In [ ]:
#| export
@patch
def save(self: MAPPO, path: str | Path, **extra: Any) -> Path:
    "Save the config, the weights (and the value normalization) and `extra` (e.g. `env_name`, `env_kwargs`) to `path`."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'config': self.config, 'state_dict': self.state_dict(), **extra}, path)
    return path


@patch(cls_method=True)
def load(cls: MAPPO, path: str | Path, map_location: Any = 'cpu') -> MAPPO:
    "A `MAPPO` saved with `save`; what it stored besides the model is in `checkpoint`."
    checkpoint = torch.load(path, map_location=map_location, weights_only=True)
    model = cls(**checkpoint.pop('config'))
    model.load_state_dict(checkpoint.pop('state_dict'))
    model.checkpoint = checkpoint
    return model.eval()

### Tests

Every critic on a small FindGoal: rollouts end when every agent is done, finished agents are
masked, a few updates run, the actor acts in a `World`, and the model loads back. Then a short
training run on the smallest FindGoal (one agent, 5 x 5) improves on its random start.

In [ ]:
#| hide
import tempfile
import comm_core as cc
from stable_marl import World
from stable_marl.policy import FeedForwardPolicy

kw = dict(agents=2, size=7, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=15)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=3, goal_conditioned=True, **kw)
world.reset(seed=[0, 1, 2])
columns = {k: v[:, -1] for k, v in world.infos.items() if not k.startswith('_')}
dim = commnet_features(columns, grid_size=7).shape[-1]
state_dim = int(np.prod(columns['state'].shape[1:]))
for critic in CRITICS:
    torch.manual_seed(0)
    model = MAPPO(dim, 2, critic=critic, state_dim=state_dim, grid_size=7, hidden_dim=32)
    batch, stats = rollout(model, world, seed=0)
    T = len(batch['action'])
    test_eq(batch['critic_input'].shape[:3], (T, 3, 2))
    test_eq(int(stats['length'].max()), T)
    for e in range(3):                                                  # an env's episode ends once, when all are done
        end = int(torch.nonzero(batch['ended'][:, e])[0])
        test_eq(end + 1, int(stats['length'][e]))
        assert not batch['alive'][end + 1:, e].any()
        if stats['success'][e]:
            test_eq(batch['terminal'][:, e].any(0).all(), True)
    assert (batch['obs'][~batch['alive']] == 0).all()                   # death masking
    logs = []
    train_mappo(model, world, episodes=6, epochs=2, log_every=1, log=logs.append)
    assert all(np.isfinite(r['loss']) for r in logs)
    world.set_policy(FeedForwardPolicy(model))
    test_eq(len(world.evaluate(episodes=3, seed=100)['episode_successes']), 3)

# the critics' observations over comm-core: a channel losing everything leaves only each agent's own
model = MAPPO(dim, 2, grid_size=7, hidden_dim=32)
model.links = NetworkLinks(2, channel=cc.ErasureChannel(loss_rate=1.0))
batch, _ = rollout(model, world, seed=0)
other = batch['critic_input'][..., dim + 1:2 * (dim + 1)]               # agent 1's slot in the critics' inputs
test_eq(other[:, :, 0].abs().sum(), torch.tensor(0.))                   # agent 0's critic never got agent 1's observation
model.links = None

path = model.save(f'{tempfile.mkdtemp()}/mappo.pt', env_name='MultiGrid-FindGoal-15x15-v0', env_kwargs=kw)
again = MAPPO.load(path)
test_eq(again.checkpoint['env_kwargs'], kw)
x = model.critic_inputs(batch['obs'][0])
test_close(again(batch['obs'][0], x)[1], model(batch['obs'][0], x)[1])

In [ ]:
#| hide
small = dict(agents=1, size=5, num_obstacles=0, n_clutter=0, min_goal_spawn_distance=1, tile_size=4, max_steps=20)
world = World('MultiGrid-FindGoal-15x15-v0', num_envs=8, goal_conditioned=True, **small)
world.reset(seed=list(range(8)))
dim = commnet_features({k: v[:, -1] for k, v in world.infos.items() if not k.startswith('_')}, grid_size=5).shape[-1]
torch.manual_seed(0)
model = MAPPO(dim, 1, grid_size=5, hidden_dim=64)
logs = []
train_mappo(model, world, episodes=1200, log_every=50, log=logs.append)
first, last = logs[0]['success_rate'], logs[-1]['success_rate']
print(f'success while training: {first:.2f} -> {last:.2f}')
assert last > first

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()